# Baseline cot_structured_sc + high_div_8 Eval

Run the base `Qwen/Qwen3-4B-Thinking-2507` model with the repo's `cot_structured_sc_only` run config. This uses `cot_structured_sc` plus `high_div_8` self-consistency and deliberately runs **without** a LoRA adapter.

## 1. GPU Check

In [ ]:
!nvidia-smi


## 2. Mount Drive and Locate Repo

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')

REPO_URL = os.environ.get('REPO_URL', 'https://github.com/DarinAnthony/151B_SP26_Competition.git')
MARKER = Path('experiments/prompt_engineering/src/eval.py')

def find_repo_root() -> Path | None:
    candidates = []
    if os.environ.get('REPO_ROOT'):
        candidates.append(Path(os.environ['REPO_ROOT']).expanduser())
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        Path('/content/151B_SP26_Competition'),
        Path('/content/drive/MyDrive/qwen_math_comp/151B_SP26_Competition'),
        Path('/content/drive/MyDrive/151B_SP26_Competition'),
    ])
    for root in candidates:
        if (root / MARKER).exists():
            return root.resolve()
    return None

REPO_ROOT = find_repo_root()
if REPO_ROOT is None and IN_COLAB:
    clone_dir = Path('/content/151B_SP26_Competition')
    if not clone_dir.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(clone_dir)], check=True)
    REPO_ROOT = find_repo_root()

if REPO_ROOT is None:
    raise FileNotFoundError(
        'Could not find the repo root. Set os.environ["REPO_ROOT"] to the '
        '151B_SP26_Competition path, or set REPO_URL to a cloneable repo URL, and rerun this cell.'
    )

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.environ['REPO_ROOT'] = str(REPO_ROOT)

# Prevent accidental adapter use through shared.runner.ADAPTER_PATH fallback.
os.environ.pop('ADAPTER_PATH', None)

print('IN_COLAB =', IN_COLAB)
print('REPO_ROOT =', REPO_ROOT)
print('ADAPTER_PATH in env =', os.environ.get('ADAPTER_PATH'))


## 3. Install Dependencies

In [ ]:
%pip install -q -U pip uv

!uv pip uninstall --system -y vllm torch torchvision torchaudio triton
!uv pip install --system --torch-backend=cu130 \
  "vllm==0.21.0" \
  "bitsandbytes>=0.46.1" \
  "jedi>=0.16" \
  "hydra-core>=1.3" \
  datasets accelerate transformers tqdm sympy pyyaml wandb


## 4. Baseline Run Config

Default target is the held-out public split, `data/test.jsonl`, with `max_tokens=16384`. Change `EVAL_DATA_PATH` to `data/public.jsonl` only if you want the full 1,126-row public-set sanity check.

In [ ]:
from pathlib import Path
import os

RESULTS_DIR = Path(os.environ.get(
    'RESULTS_DIR',
    '/content/drive/MyDrive/qwen_math_comp/eval_results' if IN_COLAB else 'experiments/prompt_engineering/results',
))
DRIVE_RESULTS_DIR = Path(os.environ.get(
    'DRIVE_RESULTS_DIR',
    '/content/drive/MyDrive/qwen_math_comp/eval_results' if IN_COLAB else str(RESULTS_DIR),
))
EVAL_DATA_PATH = Path(os.environ.get('EVAL_DATA_PATH', 'data/test.jsonl'))
HELDOUT_SLICE_N = int(os.environ.get('HELDOUT_SLICE_N', '100'))
RUN_NAME = os.environ.get('RUN_NAME', 'base_cot_structured_sc_high_div_8_heldout_16384')
MAX_TOKENS = int(os.environ.get('MAX_TOKENS', '16384'))
RUNNER_MAX_MODEL_LEN = int(os.environ.get('RUNNER_MAX_MODEL_LEN', str(max(16384, MAX_TOKENS + 4096))))
PRIVATE_BATCH_SIZE = int(os.environ.get('PRIVATE_BATCH_SIZE', '25'))
PRIVATE_N_SAMPLES = int(os.environ.get('PRIVATE_N_SAMPLES', '5'))
PRIVATE_DATA_PATH = Path(os.environ.get(
    'PRIVATE_DATA_PATH',
    '/content/drive/MyDrive/qwen_math_comp/private.jsonl' if IN_COLAB else 'data/private.jsonl',
))
SUBMISSION_DIR = Path(os.environ.get(
    'SUBMISSION_DIR',
    f'/content/drive/MyDrive/qwen_math_comp/submissions/{RUN_NAME}' if IN_COLAB else str(RESULTS_DIR / 'submissions' / RUN_NAME),
))
SUBMISSION_PATH = Path(os.environ.get('SUBMISSION_PATH', str(SUBMISSION_DIR / 'private.jsonl')))

# vLLM is the intended path. HF fallback with n=8 can be slow, so keep batches small.
RUNNER_MICRO_BATCH_SIZE = os.environ.get('RUNNER_MICRO_BATCH_SIZE', '2')
RUNNER_PARALLEL_SAMPLES = os.environ.get('RUNNER_PARALLEL_SAMPLES', '1')
RUNNER_VLLM_USE_TQDM = os.environ.get('RUNNER_VLLM_USE_TQDM', '1')
RUNNER_VERBOSE = os.environ.get('RUNNER_VERBOSE', '1')
WANDB_NAME = os.environ.get('WANDB_NAME', RUN_NAME)

if not EVAL_DATA_PATH.exists():
    raise FileNotFoundError(f'Eval data not found: {EVAL_DATA_PATH}')

os.environ['WANDB_NAME'] = WANDB_NAME
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['RUNNER_MAX_MODEL_LEN'] = str(RUNNER_MAX_MODEL_LEN)
os.environ['RUNNER_MICRO_BATCH_SIZE'] = RUNNER_MICRO_BATCH_SIZE
os.environ['RUNNER_PARALLEL_SAMPLES'] = RUNNER_PARALLEL_SAMPLES
os.environ['RUNNER_VLLM_USE_TQDM'] = RUNNER_VLLM_USE_TQDM
os.environ['RUNNER_VERBOSE'] = RUNNER_VERBOSE
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ.pop('ADAPTER_PATH', None)

print('EVAL_DATA_PATH =', EVAL_DATA_PATH)
print('HELDOUT_SLICE_N =', HELDOUT_SLICE_N)
print('RESULTS_DIR =', RESULTS_DIR)
print('DRIVE_RESULTS_DIR =', DRIVE_RESULTS_DIR)
print('RUN_NAME =', RUN_NAME)
print('MAX_TOKENS =', MAX_TOKENS)
print('RUNNER_MAX_MODEL_LEN =', RUNNER_MAX_MODEL_LEN)
print('PRIVATE_BATCH_SIZE =', PRIVATE_BATCH_SIZE)
print('PRIVATE_N_SAMPLES =', PRIVATE_N_SAMPLES)
print('PRIVATE_DATA_PATH =', PRIVATE_DATA_PATH)
print('SUBMISSION_PATH =', SUBMISSION_PATH)
print('RUNNER_MICRO_BATCH_SIZE =', RUNNER_MICRO_BATCH_SIZE)
print('RUNNER_PARALLEL_SAMPLES =', RUNNER_PARALLEL_SAMPLES)
print('RUNNER_VLLM_USE_TQDM =', RUNNER_VLLM_USE_TQDM)
print('RUNNER_VERBOSE =', RUNNER_VERBOSE)
print('ADAPTER_PATH in env =', os.environ.get('ADAPTER_PATH'))


## 5. Verify Prompt and Regime

In [ ]:
from experiments.prompt_engineering.src.prompts import PROMPTS
from experiments.prompt_engineering.src.eval import _load_named_regime

prompt = PROMPTS['cot_structured_sc']
regime = _load_named_regime('high_div_8')

print('prompt id:', prompt.id)
print('prompt kind:', prompt.kind)
print('regime:', regime)
print('\nSystem prompt for free-response:\n')
print(prompt.system_free)
print('\nSystem prompt for MCQ:\n')
print(prompt.system_mcq)


## 6. Eval Helper

In [ ]:
import os
import shlex
import subprocess
import sys
import time
from pathlib import Path


def _is_default_100_slice(slice_indices) -> bool:
    return slice_indices is not None and list(slice_indices) == list(range(100))


def build_eval_cmd(run_name: str, *, max_tokens: int = MAX_TOKENS, data_path: Path = EVAL_DATA_PATH, results_dir: Path = RESULTS_DIR, slice_indices=None) -> list[str]:
    cmd = [
        sys.executable,
        '-m', 'experiments.prompt_engineering.src.eval',
        'run=cot_structured_sc_only',
        'eval=full',
        f'eval.data_path={data_path}',
        f'eval.max_tokens={max_tokens}',
        'runner.engine=vllm',
        'runner.quant=bf16',
        'runner.adapter_path=null',
        f'results_dir={results_dir}',
        f'run_name={run_name}',
    ]
    if _is_default_100_slice(slice_indices):
        cmd.insert(5, 'eval.slice=default')
    elif slice_indices is not None:
        joined = ','.join(str(int(i)) for i in slice_indices)
        cmd.insert(5, f'eval.slice_indices=[{joined}]')
    return cmd


def run_eval(run_name: str, *, max_tokens: int = MAX_TOKENS, data_path: Path = EVAL_DATA_PATH, results_dir: Path = RESULTS_DIR, slice_indices=None, micro_batch_size=None) -> None:
    env = os.environ.copy()
    env.pop('ADAPTER_PATH', None)
    env['WANDB_NAME'] = run_name
    env['TOKENIZERS_PARALLELISM'] = 'false'
    env['PYTHONUNBUFFERED'] = '1'
    env['RUNNER_MAX_MODEL_LEN'] = str(RUNNER_MAX_MODEL_LEN)
    env['EVAL_METRICS_DIR'] = str(DRIVE_RESULTS_DIR)
    env['RUNNER_MICRO_BATCH_SIZE'] = str(micro_batch_size or RUNNER_MICRO_BATCH_SIZE)
    env['RUNNER_PARALLEL_SAMPLES'] = RUNNER_PARALLEL_SAMPLES
    env['RUNNER_VLLM_USE_TQDM'] = RUNNER_VLLM_USE_TQDM
    env['RUNNER_VERBOSE'] = RUNNER_VERBOSE

    cmd = build_eval_cmd(
        run_name,
        max_tokens=max_tokens,
        data_path=data_path,
        results_dir=results_dir,
        slice_indices=slice_indices,
    )

    print('EVAL_DATA_PATH =', data_path)
    print('ADAPTER_PATH in subprocess =', env.get('ADAPTER_PATH'))
    print('WANDB_NAME =', env['WANDB_NAME'])
    print('DRIVE_RESULTS_DIR =', env['EVAL_METRICS_DIR'])
    print('RUNNER_MAX_MODEL_LEN =', env['RUNNER_MAX_MODEL_LEN'])
    print('RUNNER_MICRO_BATCH_SIZE =', env['RUNNER_MICRO_BATCH_SIZE'])
    print('RUNNER_PARALLEL_SAMPLES =', env['RUNNER_PARALLEL_SAMPLES'])
    print('RUNNER_VLLM_USE_TQDM =', env['RUNNER_VLLM_USE_TQDM'])
    print('RUNNER_VERBOSE =', env['RUNNER_VERBOSE'])
    print('Command:', ' '.join(shlex.quote(part) for part in cmd))

    start = time.time()
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='')
    return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    print(f'Elapsed: {(time.time() - start) / 60:.1f} min')


## 7. 5-Item Smoke Test

In [ ]:
run_eval(
    run_name=f'{RUN_NAME}_smoke5',
    slice_indices=[0, 1, 2, 3, 4],
    micro_batch_size=1,
)


## 8. 100-Item Held-Out Baseline Eval

This evaluates the first `HELDOUT_SLICE_N` rows in `EVAL_DATA_PATH`. With the default settings, that is 100 rows from the 226-row held-out split at `data/test.jsonl`.


In [ ]:
heldout_slice_indices = list(range(HELDOUT_SLICE_N))
run_eval(
    f'{RUN_NAME}_first{HELDOUT_SLICE_N}',
    slice_indices=heldout_slice_indices,
)


## 9. Optional Full Held-Out Eval

This evaluates all 226 rows in `data/test.jsonl`. Use it only after the 100-row run looks worth scaling.


In [ ]:
RUN_FULL_HELDOUT = False

if RUN_FULL_HELDOUT:
    run_eval(RUN_NAME)
else:
    print('Set RUN_FULL_HELDOUT = True to evaluate all 226 held-out rows.')


## 10. Optional Full Public Eval

This includes the 900 training-split rows, so use it only to compare with older full-public logs.

In [ ]:
RUN_FULL_PUBLIC = False

if RUN_FULL_PUBLIC:
    run_eval(
        run_name='base_cot_structured_sc_high_div_8_full_public_16384',
        data_path=Path('data/public.jsonl'),
        max_tokens=MAX_TOKENS,
    )
else:
    print('Set RUN_FULL_PUBLIC = True to evaluate all 1,126 public rows.')


## 11. Private Submission Generation

This path does not score answers because the private set has no gold labels. It uses the same `cot_structured_sc` prompt and high-div sampling settings, but defaults to `PRIVATE_N_SAMPLES=5` for private generation. It majority-votes the extracted answers and writes one JSON object per line with `id`, `is_mcq`, and `response`.


In [ ]:
from dataclasses import replace

from shared.io import load_jsonl, save_jsonl
from shared.prompt_format import build_chat_messages
from shared.runner import load_model
from shared.schemas import RunnerCfg
from shared.scoring import _extract_letter, _last_boxed_content
from shared.voting import vote_index
from experiments.prompt_engineering.src.eval import _load_named_regime
from experiments.prompt_engineering.src.prompts import PROMPTS


def _extract_private_answer(item, response):
    if item.get('options'):
        return _extract_letter(response)
    return _last_boxed_content(response).strip()


def generate_private_submission(
    *,
    private_path=PRIVATE_DATA_PATH,
    output_path=SUBMISSION_PATH,
    max_tokens=MAX_TOKENS,
    batch_size=PRIVATE_BATCH_SIZE,
    slice_indices=None,
    resume=True,
):
    private_path = Path(private_path)
    output_path = Path(output_path)
    if not private_path.exists():
        raise FileNotFoundError(
            f'Private data not found: {private_path}. Set PRIVATE_DATA_PATH to the uploaded private JSONL.'
        )
    if private_path.resolve() == output_path.resolve():
        raise ValueError('Refusing to overwrite the private input file. Choose a different SUBMISSION_PATH.')

    items = load_jsonl(private_path)
    if slice_indices is not None:
        items = [items[i] for i in slice_indices if 0 <= i < len(items)]
    if not items:
        raise ValueError(f'No private items loaded from {private_path}.')
    batch_size = max(1, int(batch_size))

    completed = {}
    if resume and output_path.exists():
        for row in load_jsonl(output_path):
            if 'id' in row and 'response' in row:
                completed[int(row['id'])] = row
        print(f'Resuming from {output_path}: {len(completed)} completed rows found.')

    prompt = PROMPTS['cot_structured_sc']
    sampling = replace(
        _load_named_regime('high_div_8'),
        name=f'high_div_{PRIVATE_N_SAMPLES}',
        n_samples=PRIVATE_N_SAMPLES,
    )
    runner_cfg = RunnerCfg(engine='vllm', quant='bf16', adapter_path=None)

    print(f'Loaded {len(items)} private items from {private_path}')
    print(f'Prompt={prompt.id}, regime={sampling.name}, n_samples={sampling.n_samples}, max_tokens={max_tokens}')
    global PRIVATE_HANDLE
    if 'PRIVATE_HANDLE' not in globals():
        print('Loading base model with no adapter...')
        PRIVATE_HANDLE = load_model(runner_cfg)
        print('Model loaded.')
    else:
        print('Reusing already-loaded base model.')

    pending = [item for item in items if int(item['id']) not in completed]
    print(f'Generating private responses in chunks of {batch_size}: {len(pending)} pending / {len(items)} total')

    output_path.parent.mkdir(parents=True, exist_ok=True)
    total_chunks = (len(pending) + batch_size - 1) // batch_size
    for chunk_idx, start in enumerate(range(0, len(pending), batch_size), start=1):
        chunk = pending[start:start + batch_size]
        print(f'Chunk {chunk_idx}/{total_chunks}: generating {len(chunk)} items...')
        messages = [build_chat_messages(item, prompt) for item in chunk]
        outputs = PRIVATE_HANDLE.generate_batch(messages, sampling, max_tokens)
        for item, out in zip(chunk, outputs):
            extracts = [_extract_private_answer(item, response) for response in out.responses]
            idx = vote_index(extracts)
            completed[int(item['id'])] = {
                'id': int(item['id']),
                'is_mcq': bool(item.get('options')),
                'response': out.responses[idx],
            }
        rows = [completed[int(item['id'])] for item in items if int(item['id']) in completed]
        save_jsonl(rows, output_path)
        print(f'  saved {len(rows)}/{len(items)} rows to {output_path}')

    rows = [completed[int(item['id'])] for item in items if int(item['id']) in completed]
    save_jsonl(rows, output_path)
    print(f'Saved {len(rows)} private submission rows to {output_path}')
    return output_path


## 12. 5-Item Private Generation Smoke


In [ ]:
generate_private_submission(
    output_path=SUBMISSION_PATH.with_name('private_smoke5.jsonl'),
    slice_indices=[0, 1, 2, 3, 4],
)


## 13. Full Private Submission


In [ ]:
# Run this after the private smoke file looks good.
generate_private_submission()
